# QEVD Preprocessing (Refactored)

This notebook preprocesses videos into `(1, 3, 16, 112, 112)` float32 tensors in `[0,1]`, writes `manifest.jsonl`, and is safe for Kaggle 20GB limits.

In [ ]:
!pip install -q decord scikit-learn tqdm

In [ ]:
import os
from collections import Counter, defaultdict

ROOT = "/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2"

def gb(x):
    return x / (1024**3)

print("=== BASIC ===")
print("Exists:", os.path.exists(ROOT))
print("Is dir:", os.path.isdir(ROOT))
if not os.path.isdir(ROOT):
    raise SystemExit("Folder not found.")

print("\n=== TOP LEVEL ===")
for name in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, name)
    print(("DIR " if os.path.isdir(p) else "FILE"), name)

# pick data root
DATA_ROOT = os.path.join(ROOT, "qevd_final") if os.path.isdir(os.path.join(ROOT, "qevd_final")) else ROOT
print("\nDATA_ROOT:", DATA_ROOT)

print("\n=== RECURSIVE SIZE + EXTENSIONS ===")
total_bytes = 0
total_files = 0
ext_counts = Counter()

for r, _, files in os.walk(ROOT):
    for fn in files:
        fp = os.path.join(r, fn)
        try:
            sz = os.path.getsize(fp)
        except Exception:
            continue
        total_bytes += sz
        total_files += 1
        ext = os.path.splitext(fn)[1].lower() or "<no_ext>"
        ext_counts[ext] += 1

print("Total files:", total_files)
print(f"Total size : {gb(total_bytes):.3f} GB")
print("Top extensions:")
for ext, cnt in ext_counts.most_common(10):
    print(f"  {ext:10} {cnt}")

print("\n=== CLASS AUDIT ===")
class_dirs = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])
print("Class folders:", len(class_dirs))

per_class_count = {}
per_class_bytes = {}
empty_classes = []
non_mp4_total = 0

for cls in class_dirs:
    cdir = os.path.join(DATA_ROOT, cls)
    c_count = 0
    c_bytes = 0
    for fn in os.listdir(cdir):
        fp = os.path.join(cdir, fn)
        if not os.path.isfile(fp):
            continue
        c_count += 1
        try:
            c_bytes += os.path.getsize(fp)
        except Exception:
            pass
        if not fn.lower().endswith(".mp4"):
            non_mp4_total += 1
    per_class_count[cls] = c_count
    per_class_bytes[cls] = c_bytes
    if c_count == 0:
        empty_classes.append(cls)

print("Empty classes:", len(empty_classes))
if empty_classes:
    print("Examples:", empty_classes[:10])

print("\nTop 10 classes by count:")
for cls, n in sorted(per_class_count.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {cls:40} {n:6} files  {gb(per_class_bytes[cls]):8.3f} GB")

print("\nBottom 10 classes by count:")
for cls, n in sorted(per_class_count.items(), key=lambda x: x[1])[:10]:
    print(f"  {cls:40} {n:6} files  {gb(per_class_bytes[cls]):8.3f} GB")

print("\nNon-mp4 files inside class folders:", non_mp4_total)

print("\n=== DUPLICATE BASENAMES ACROSS CLASSES ===")
name_to_classes = defaultdict(set)
for cls in class_dirs:
    cdir = os.path.join(DATA_ROOT, cls)
    for fn in os.listdir(cdir):
        fp = os.path.join(cdir, fn)
        if os.path.isfile(fp):
            name_to_classes[fn].add(cls)

dupes = [(fn, sorted(list(v))) for fn, v in name_to_classes.items() if len(v) > 1]
print("Duplicates:", len(dupes))
if dupes:
    print("Examples:")
    for fn, cls_list in dupes[:10]:
        print(" ", fn, "->", cls_list[:5], ("..." if len(cls_list) > 5 else ""))

print("\nAudit print complete.")

In [ ]:
import os, json

DATA_ROOT = "/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2/qevd_final"
EXPECTED_JSON = "/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2/26LPCVC_Track2_Sample_Solution/fine_grained_label_names_release.json"

def norm(x: str) -> str:
    return x.strip().replace(" ", "_").replace("/", "-")

observed = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])

with open(EXPECTED_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

if isinstance(raw, dict):
    expected_raw = [str(v) for v in raw.values()]
else:
    expected_raw = [str(v) for v in raw]

expected = sorted([norm(x) for x in expected_raw])

missing = sorted(set(expected) - set(observed))
extra = sorted(set(observed) - set(expected))

print("Expected(normalized):", len(expected))
print("Observed:", len(observed))
print("True missing:", len(missing), missing)
print("True extra:", len(extra), extra)

In [ ]:
import os
import json

# 1) observed in your dataset folder
observed_classes = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

# 2) expected full list (92 classes)
# Option A: set this to sample repo json if you have it
expected_json = "/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2/26LPCVC_Track2_Sample_Solution/class_labels.json"

# Option B: if you already copied/pointed SAMPLE_CLASS_LABELS_JSON, use that

if not expected_json or not os.path.exists(expected_json):
    raise FileNotFoundError(expected_json)

with open(expected_json, "r", encoding="utf-8") as f:
    raw = json.load(f)

# normalize to class-name list
if isinstance(raw, dict):
    expected_classes = sorted([str(v) for v in raw.values()])
else:
    expected_classes = sorted([str(x) for x in raw])

missing_classes = sorted(set(expected_classes) - set(observed_classes))
extra_classes   = sorted(set(observed_classes) - set(expected_classes))

print("Expected classes:", len(expected_classes))
print("Observed classes:", len(observed_classes))
print("Missing classes :", len(missing_classes))
print("Extra classes   :", len(extra_classes))

print("\nMissing class names:")
for c in missing_classes:
    print(" -", c)

if extra_classes:
    print("\nExtra class names (in data but not expected list):")
    for c in extra_classes:
        print(" -", c)

In [ ]:
# Build class mapping with robust name normalization (official 92-name labels)
import os
import json
import re
from sklearn.model_selection import train_test_split

OUT_ROOT = "/kaggle/working/remapped_classes"
os.makedirs(OUT_ROOT, exist_ok=True)

# Point to official label file (list or dict accepted)
SAMPLE_CLASS_LABELS_JSON = "/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2/26LPCVC_Track2_Sample_Solution/class_labels.json"

assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT}"
assert os.path.exists(SAMPLE_CLASS_LABELS_JSON), f"Official labels file not found: {SAMPLE_CLASS_LABELS_JSON}"

with open(SAMPLE_CLASS_LABELS_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

if isinstance(raw, list):
    official_classes = [x for x in raw if isinstance(x, str)]
elif isinstance(raw, dict):
    # Supports {"0":"class_a", ...} or {"class_a":0, ...}
    if all(isinstance(k, str) and isinstance(v, int) for k, v in raw.items()):
        official_classes = [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    else:
        official_classes = [v for _, v in sorted(raw.items(), key=lambda kv: int(kv[0])) if isinstance(v, str)]
else:
    raise ValueError("Unsupported label file format. Expected list[str] or dict.")

observed_dirs = sorted([
    d for d in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, d))
])

def norm_name(s: str) -> str:
    s = s.strip().lower().replace("_", " ")
    s = re.sub(r"\s+", " ", s)
    return s

official_by_norm = {}
for name in official_classes:
    k = norm_name(name)
    if k in official_by_norm and official_by_norm[k] != name:
        raise RuntimeError(f"Official label normalization collision: '{official_by_norm[k]}' vs '{name}'")
    official_by_norm[k] = name

dir_to_official = {}
unmapped_dirs = []
for d in observed_dirs:
    k = norm_name(d)
    if k in official_by_norm:
        dir_to_official[d] = official_by_norm[k]
    else:
        unmapped_dirs.append(d)

mapped_official = set(dir_to_official.values())
missing_official = [c for c in official_classes if c not in mapped_official]

print(f"Expected classes: {len(official_classes)}")
print(f"Observed dirs:    {len(observed_dirs)}")
print(f"Mapped dirs:      {len(dir_to_official)}")
print(f"Unmapped dirs:    {len(unmapped_dirs)}")
print(f"Missing classes:  {len(missing_official)}")
if unmapped_dirs:
    print("Unmapped dir examples:", unmapped_dirs[:10])
if missing_official:
    print("Missing class examples:", missing_official[:10])

# Keep class_map/class_labels in official order/names
classes = official_classes
class_map = {name: i for i, name in enumerate(classes)}
class_labels = {str(i): name for i, name in enumerate(classes)}

with open(os.path.join(OUT_ROOT, "class_map.json"), "w", encoding="utf-8") as f:
    json.dump(class_map, f, indent=2)
with open(os.path.join(OUT_ROOT, "class_labels.json"), "w", encoding="utf-8") as f:
    json.dump(class_labels, f, indent=2)
with open(os.path.join(OUT_ROOT, "missing_expected_classes.json"), "w", encoding="utf-8") as f:
    json.dump(missing_official, f, indent=2)
with open(os.path.join(OUT_ROOT, "unmapped_observed_dirs.json"), "w", encoding="utf-8") as f:
    json.dump(unmapped_dirs, f, indent=2)

# Build split from observed dirs, but labels use official names
vids_list, labels_list = [], []
for d in observed_dirs:
    if d not in dir_to_official:
        continue
    official_label = dir_to_official[d]
    cls_dir = os.path.join(DATA_ROOT, d)
    vids = sorted([f for f in os.listdir(cls_dir) if f.endswith(".mp4")])
    if not vids:
        continue
    if len(vids) == 1:
        rel = os.path.join(d, vids[0])  # keep real folder path
        vids_list.extend([rel, rel])    # duplicate so stratify does not fail
        labels_list.extend([official_label, official_label])
    else:
        for f in vids:
            vids_list.append(os.path.join(d, f))
            labels_list.append(official_label)

assert vids_list, "No videos found after mapping."

train_files, val_files = train_test_split(
    vids_list,
    test_size=0.15,
    stratify=labels_list,
    random_state=42
)
split_files = {"train": train_files, "val": val_files}
print(f"Split complete: train={len(train_files)}, val={len(val_files)}")

In [ ]:
import os, json, time, glob
import numpy as np
import torch
import torchvision.transforms as T
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from decord import VideoReader, cpu

print('Imports ready')

In [ ]:
# Kaggle-friendly config
DATASET_INPUT = '/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2'
DATA_ROOT = os.path.join(DATASET_INPUT, 'qevd_final')
OUT_ROOT = '/kaggle/working/preprocessed_tensors'

CLIP_LEN = 16
MAX_STORAGE_GB = 18.0

# Reserve validation budget by ratio while still guaranteeing class coverage.
RESERVE_VAL_BUDGET = True
TARGET_VAL_RATIO = 0.15

# Class sampling policy:
# - False: use all available videos (recommended), bounded by MAX_STORAGE_GB
# - True: limit videos per class per split using CLASS_CAPS
USE_CLASS_CAPS = False
CLASS_CAPS = {'train': 75, 'val': 15}

# Minority-class augmentation (applies only to train split)
ENABLE_MINORITY_AUGMENT = True
MIN_TRAIN_SAMPLES_PER_CLASS = 24
MAX_AUG_PER_VIDEO = 2

# Competition safety switches
EXPECTED_NUM_CLASSES = 92
COMPETITION_SAFE_MODE = True

# Optional: manual override path to official class names json
SAMPLE_CLASS_LABELS_JSON = ''
CLASS_LABELS_CANDIDATES = [
    SAMPLE_CLASS_LABELS_JSON,
    '/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2/26LPCVC_Track2_Sample_Solution/fine_grained_label_names_release.json',
    '/kaggle/input/notebooks/dadhichisarkershayon/qevd-dataset-creation-part-2/26LPCVC_Track2_Sample_Solution/class_labels.json',
    '/kaggle/input/qevd-sample-solution/class_labels.json',
    '/kaggle/input/lpcvc-track2-sample-solution/class_labels.json',
    '/kaggle/input/qevd-preprocessed/class_labels.json',
]

assert os.path.isdir(DATA_ROOT), f'DATA_ROOT not found: {DATA_ROOT}'
os.makedirs(OUT_ROOT, exist_ok=True)
print('Paths validated')

In [ ]:
# Build class mapping and split with robust folder-name -> official-name normalization
import re

observed_dirs = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])

def norm_name(s: str) -> str:
    s = s.strip().lower().replace('_', ' ')
    s = re.sub(r'\s+', ' ', s)
    return s

resolved_labels_path = next((p for p in CLASS_LABELS_CANDIDATES if p and os.path.exists(p)), '')
if COMPETITION_SAFE_MODE and not resolved_labels_path:
    raise RuntimeError(
        'Competition-safe mode requires an official class list file, but none was found. '
        'Set SAMPLE_CLASS_LABELS_JSON to the official 92-class labels JSON path.'
    )

if resolved_labels_path:
    with open(resolved_labels_path, 'r', encoding='utf-8') as f:
        raw = json.load(f)

    if isinstance(raw, list):
        full_classes = [x for x in raw if isinstance(x, str)]
    elif isinstance(raw, dict):
        # Supports {"0":"class_a", ...} or {"class_a":0, ...}
        if raw and all(isinstance(k, str) and isinstance(v, int) for k, v in raw.items()):
            full_classes = [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
        else:
            full_classes = [
                v for _, v in sorted(raw.items(), key=lambda kv: int(kv[0]))
                if isinstance(v, str)
            ]
    else:
        raise ValueError('Unsupported class label file format')

    classes = full_classes
    print(f'Using full class list file: {resolved_labels_path}')
    print(f'Full class list size: {len(classes)} classes')
else:
    classes = observed_dirs
    print(f'Using observed classes from dataset: {len(classes)} classes (no full class list file found)')

official_by_norm = {}
for name in classes:
    k = norm_name(name)
    if k in official_by_norm and official_by_norm[k] != name:
        raise RuntimeError(f"Class normalization collision: '{official_by_norm[k]}' vs '{name}'")
    official_by_norm[k] = name

dir_to_official = {}
unmapped_dirs = []
for d in observed_dirs:
    k = norm_name(d)
    if k in official_by_norm:
        dir_to_official[d] = official_by_norm[k]
    else:
        unmapped_dirs.append(d)

if COMPETITION_SAFE_MODE and len(classes) != EXPECTED_NUM_CLASSES:
    raise RuntimeError(
        f'Competition-safe mode requires {EXPECTED_NUM_CLASSES} classes in class_map, got {len(classes)}. '
        'Use the official 92-class labels JSON.'
    )
elif len(classes) != EXPECTED_NUM_CLASSES:
    print(
        f'Warning: class_map size is {len(classes)} (expected {EXPECTED_NUM_CLASSES}). '
        'This can break submission format if evaluator expects all classes.'
    )

missing_from_data = sorted([c for c in classes if c not in set(dir_to_official.values())])
if missing_from_data:
    print(f'Classes missing in DATA_ROOT but kept in class_map: {len(missing_from_data)}')
    print('Examples:', missing_from_data[:10])
if unmapped_dirs:
    print(f'Observed dirs not mapped to class list: {len(unmapped_dirs)}')
    print('Examples:', unmapped_dirs[:10])

# Collect videos for split from observed dirs, but labels use official names.
# Single-video classes are assigned to train only to prevent train/val leakage.
split_candidates, split_labels = [], []
single_video_train_only = []

for d in observed_dirs:
    if d not in dir_to_official:
        continue
    official_label = dir_to_official[d]
    cls_dir = os.path.join(DATA_ROOT, d)
    vids = sorted([f for f in os.listdir(cls_dir) if f.endswith('.mp4')])
    if not vids:
        continue

    if len(vids) == 1:
        single_video_train_only.append(os.path.join(d, vids[0]))
    else:
        for f in vids:
            split_candidates.append(os.path.join(d, f))
            split_labels.append(official_label)

if not split_candidates and not single_video_train_only:
    raise RuntimeError('No videos found after mapping; check DATA_ROOT and naming.')

class_map = {name: i for i, name in enumerate(classes)}
class_labels = {str(i): name for i, name in enumerate(classes)}

with open(os.path.join(OUT_ROOT, 'class_map.json'), 'w', encoding='utf-8') as f:
    json.dump(class_map, f, indent=2)
with open(os.path.join(OUT_ROOT, 'class_labels.json'), 'w', encoding='utf-8') as f:
    json.dump(class_labels, f, indent=2)
with open(os.path.join(OUT_ROOT, 'missing_expected_classes.json'), 'w', encoding='utf-8') as f:
    json.dump(missing_from_data, f, indent=2)

if split_candidates:
    if len(set(split_labels)) >= 2:
        train_files, val_files = train_test_split(
            split_candidates, test_size=0.15, stratify=split_labels, random_state=42
        )
    else:
        train_files, val_files = train_test_split(
            split_candidates, test_size=0.15, random_state=42
        )
else:
    train_files, val_files = [], []

train_files.extend(single_video_train_only)

overlap = set(train_files) & set(val_files)
if overlap:
    raise RuntimeError(
        f'Data leakage detected: {len(overlap)} videos are present in both train and val. '
        f'Examples: {sorted(overlap)[:10]}'
    )

split_files = {'train': train_files, 'val': val_files}
print(f'Split complete: train={len(train_files)}, val={len(val_files)}')
print(f'Single-video samples forced to train-only: {len(single_video_train_only)}')

In [ ]:
from collections import Counter, deque, defaultdict
from tqdm.auto import tqdm
import time

spatial = T.Compose([
    T.ConvertImageDtype(torch.float32),
    T.Resize((128, 171), antialias=False),
    T.CenterCrop((112, 112)),
])

def _safe_indices(total_frames, clip_len):
    # Full-video coverage: sample clip_len points uniformly from start to end.
    if total_frames <= 0:
        return []
    if total_frames == 1:
        return [0] * clip_len

    idx = np.linspace(0, total_frames - 1, num=clip_len, dtype=np.int64).tolist()
    return idx

def _augment_clip_for_minority(arr, rng):
    # arr shape: [1, C, T, H, W], float32 in [0,1]
    aug = arr.copy()

    # Horizontal flip (spatial augmentation)
    if rng.random() < 0.5:
        aug = aug[:, :, :, :, ::-1].copy()

    # Mild temporal roll to vary frame alignment
    shift = int(rng.integers(-2, 3))
    if shift != 0:
        aug = np.roll(aug, shift=shift, axis=2)

    # Mild brightness scaling
    gain = float(rng.uniform(0.85, 1.15))
    aug = np.clip(aug * gain, 0.0, 1.0)

    # Low Gaussian noise
    sigma = float(rng.uniform(0.0, 0.01))
    if sigma > 0.0:
        noise = rng.normal(0.0, sigma, size=aug.shape).astype(np.float32)
        aug = np.clip(aug + noise, 0.0, 1.0)

    return aug.astype(np.float32, copy=False)

# Keep logs readable in Kaggle: print only first N decode failures
MAX_DEBUG_DECODE_PRINTS = 15
decode_printed = 0
decode_error_types = Counter()

def extract_clip(video_path):
    global decode_printed
    try:
        vr = VideoReader(video_path, ctx=cpu(0))
        total = len(vr)
        if total <= 0:
            decode_error_types['empty_video'] += 1
            return None

        idx = _safe_indices(total, CLIP_LEN)

        frames = vr.get_batch(idx).asnumpy()  # [T,H,W,C], uint8
        frames = torch.from_numpy(frames)

        clip = frames.permute(0, 3, 1, 2)     # [T,C,H,W]
        # ConvertImageDtype already scales uint8 to float32 [0, 1]
        clip = spatial(clip)
        clip = clip.permute(1, 0, 2, 3).unsqueeze(0).contiguous()  # [1,C,T,H,W]
        return clip.to(torch.float32).numpy()
    except Exception as e:
        decode_error_types[type(e).__name__] += 1
        if decode_printed < MAX_DEBUG_DECODE_PRINTS:
            print(f"[decode fail {decode_printed+1}/{MAX_DEBUG_DECODE_PRINTS}] {video_path} -> {e}")
            decode_printed += 1
        return None

# Build minority augmentation plan from train split counts
train_counts = Counter()
for rel_vpath in split_files.get('train', []):
    cls_dirname = os.path.dirname(rel_vpath)
    cls = dir_to_official.get(cls_dirname, cls_dirname)
    train_counts[cls] += 1

augment_remaining = {}
if ENABLE_MINORITY_AUGMENT:
    for cls, count in train_counts.items():
        need = max(0, MIN_TRAIN_SAMPLES_PER_CLASS - count)
        if need > 0:
            augment_remaining[cls] = need

    print(
        f"Minority augmentation plan: {len(augment_remaining)} classes below "
        f"{MIN_TRAIN_SAMPLES_PER_CLASS} train samples"
    )
    if augment_remaining:
        total_aug_planned = sum(augment_remaining.values())
        print(f"Planned augmented tensors (upper bound): {total_aug_planned}")
else:
    print('Minority augmentation disabled')

rng = np.random.default_rng(42)
augmented_written = 0
augmented_written_by_class = Counter()

manifest_path = os.path.join(OUT_ROOT, 'manifest.jsonl')
max_bytes = int(MAX_STORAGE_GB * 1024 ** 3)
used_bytes = 0

processed = 0
skipped = 0
processed_by_split = Counter()
skipped_by_split = Counter()

TENSOR_BYTES_ESTIMATE = CLIP_LEN * 3 * 112 * 112 * 4

overall_start = time.time()

# Reserve validation budget so val has both class coverage and useful volume.
val_classes = set()
for rel_vpath in split_files.get('val', []):
    cls_dirname = os.path.dirname(rel_vpath)
    cls = dir_to_official.get(cls_dirname, cls_dirname)
    val_classes.add(cls)

total_split_items = len(split_files.get('train', [])) + len(split_files.get('val', []))
observed_val_ratio = len(split_files.get('val', [])) / max(total_split_items, 1)
effective_val_ratio = max(TARGET_VAL_RATIO, observed_val_ratio)

val_min_required_bytes = len(val_classes) * TENSOR_BYTES_ESTIMATE
val_ratio_reserved_bytes = int(max_bytes * effective_val_ratio)
val_reserved_bytes = max(val_min_required_bytes, val_ratio_reserved_bytes)

if RESERVE_VAL_BUDGET and val_classes:
    if max_bytes <= val_min_required_bytes:
        raise RuntimeError(
            f"Storage cap too low. Need > {val_min_required_bytes / (1024 ** 3):.2f} GB "
            f"to reserve at least 1 val tensor per class, but max is {max_bytes / (1024 ** 3):.2f} GB."
        )
    val_reserved_bytes = min(val_reserved_bytes, max_bytes - TENSOR_BYTES_ESTIMATE)
    train_write_limit_bytes = max_bytes - val_reserved_bytes
    print(
        f"Val reserve enabled: ratio target={effective_val_ratio:.3f}, classes={len(val_classes)} -> "
        f"reserve={val_reserved_bytes / (1024 ** 3):.2f} GB (min={val_min_required_bytes / (1024 ** 3):.2f} GB); "
        f"train budget {train_write_limit_bytes / (1024 ** 3):.2f} GB"
    )
else:
    train_write_limit_bytes = max_bytes

# Build rough total attempts for overall progress bar (base videos only)
estimated_total_attempts = 0
for split in ('train', 'val'):
    tmp_by_class = defaultdict(list)
    for rel_vpath in split_files[split]:
        cls_dirname = os.path.dirname(rel_vpath)
        cls = dir_to_official.get(cls_dirname, cls_dirname)
        tmp_by_class[cls].append(rel_vpath)

    if USE_CLASS_CAPS:
        estimated_total_attempts += sum(min(len(v), CLASS_CAPS[split]) for v in tmp_by_class.values())
    else:
        estimated_total_attempts += sum(len(v) for v in tmp_by_class.values())

with open(manifest_path, 'w', encoding='utf-8') as manifest, tqdm(
    total=estimated_total_attempts,
    desc='Overall',
    unit='video',
    dynamic_ncols=True
) as pbar_overall:

    for split in ('train', 'val'):
        split_start = time.time()
        train_stopped_for_val_budget = False

        files_by_class = {}
        for rel_vpath in split_files[split]:
            cls_dirname = os.path.dirname(rel_vpath)
            cls = dir_to_official.get(cls_dirname, cls_dirname)
            files_by_class.setdefault(cls, []).append(rel_vpath)

        classes = sorted(files_by_class.keys())
        min_required_bytes = len(classes) * TENSOR_BYTES_ESTIMATE
        if split == 'val' and max_bytes < min_required_bytes:
            raise RuntimeError(
                f"Storage cap too low for class coverage in split '{split}'. "
                f"Need at least {min_required_bytes / (1024 ** 3):.2f} GB for one tensor/class, "
                f"but max is {max_bytes / (1024 ** 3):.2f} GB."
            )

        rr_queues = {}
        for cls, paths in files_by_class.items():
            sorted_paths = sorted(paths)
            if USE_CLASS_CAPS:
                rr_queues[cls] = deque(sorted_paths[:CLASS_CAPS[split]])
            else:
                rr_queues[cls] = deque(sorted_paths)

        active_classes = deque([cls for cls in classes if rr_queues[cls]])
        processed_by_class = Counter()
        uncovered_classes = set(active_classes)

        split_attempt_budget = sum(len(rr_queues[c]) for c in rr_queues)

        with tqdm(
            total=split_attempt_budget,
            desc=f'{split}',
            unit='video',
            dynamic_ncols=True,
            leave=True
        ) as pbar_split:

            last_heartbeat = time.time()
            HEARTBEAT_SECS = 20

            while active_classes:
                if used_bytes >= max_bytes:
                    break

                if split == 'train' and RESERVE_VAL_BUDGET and used_bytes >= train_write_limit_bytes:
                    train_stopped_for_val_budget = True
                    break

                cls = active_classes.popleft()
                if not rr_queues[cls]:
                    continue

                rel_vpath = rr_queues[cls].popleft()
                cls_out = os.path.join(OUT_ROOT, split, cls)
                os.makedirs(cls_out, exist_ok=True)

                full_vpath = os.path.join(DATA_ROOT, rel_vpath)
                arr = extract_clip(full_vpath)

                if arr is None:
                    skipped += 1
                    skipped_by_split[split] += 1
                else:
                    stem = os.path.splitext(os.path.basename(rel_vpath))[0]
                    out_path = os.path.join(cls_out, f'{stem}.npy')
                    if os.path.exists(out_path):
                        dup_idx = 1
                        while True:
                            candidate = os.path.join(cls_out, f'{stem}__dup{dup_idx}.npy')
                            if not os.path.exists(candidate):
                                out_path = candidate
                                break
                            dup_idx += 1

                    np.save(out_path, arr)
                    file_size = os.path.getsize(out_path)
                    used_bytes += file_size

                    manifest.write(json.dumps({
                        'label': cls,
                        'split': split,
                        'source_video': rel_vpath,
                        'tensor_path': out_path,
                        'shape': list(arr.shape),
                        'dtype': 'float32',
                        'is_augmented': False
                    }) + '\n')

                    processed += 1
                    processed_by_split[split] += 1
                    processed_by_class[cls] += 1
                    uncovered_classes.discard(cls)

                    # Add train-only synthetic tensors for minority classes
                    if split == 'train' and ENABLE_MINORITY_AUGMENT and augment_remaining.get(cls, 0) > 0:
                        extra_to_make = min(MAX_AUG_PER_VIDEO, augment_remaining[cls])
                        for _ in range(extra_to_make):
                            if used_bytes >= max_bytes:
                                break
                            if RESERVE_VAL_BUDGET and used_bytes >= train_write_limit_bytes:
                                train_stopped_for_val_budget = True
                                break

                            aug_arr = _augment_clip_for_minority(arr, rng)
                            aug_idx = augmented_written_by_class[cls] + 1
                            aug_path = os.path.join(cls_out, f'{stem}__aug{aug_idx}.npy')
                            if os.path.exists(aug_path):
                                dup_idx = 1
                                while True:
                                    candidate = os.path.join(cls_out, f'{stem}__aug{aug_idx}__dup{dup_idx}.npy')
                                    if not os.path.exists(candidate):
                                        aug_path = candidate
                                        break
                                    dup_idx += 1

                            np.save(aug_path, aug_arr)
                            aug_file_size = os.path.getsize(aug_path)
                            used_bytes += aug_file_size

                            manifest.write(json.dumps({
                                'label': cls,
                                'split': split,
                                'source_video': rel_vpath,
                                'tensor_path': aug_path,
                                'shape': list(aug_arr.shape),
                                'dtype': 'float32',
                                'is_augmented': True,
                                'augmentation_kind': 'minority_class_boost'
                            }) + '\n')

                            processed += 1
                            processed_by_split[split] += 1
                            processed_by_class[cls] += 1
                            augmented_written += 1
                            augmented_written_by_class[cls] += 1
                            augment_remaining[cls] -= 1

                # Requeue class if it still has work and budget remains
                if rr_queues[cls] and used_bytes < max_bytes:
                    active_classes.append(cls)

                # Update both progress bars (base-video attempts)
                pbar_split.update(1)
                pbar_overall.update(1)

                storage_gb = used_bytes / (1024 ** 3)
                postfix = {
                    'ok': processed,
                    'skip': skipped,
                    'aug': augmented_written,
                    'storage_gb': f'{storage_gb:.2f}/{MAX_STORAGE_GB:.1f}',
                    'covered': f'{len(processed_by_class)}/{len(classes)}'
                }
                pbar_split.set_postfix(postfix)
                pbar_overall.set_postfix(postfix)

                # Periodic heartbeat
                now = time.time()
                if now - last_heartbeat >= HEARTBEAT_SECS:
                    elapsed = now - overall_start
                    rate = (processed + skipped) / max(elapsed, 1e-6)
                    remaining = max(0, estimated_total_attempts - (processed + skipped))
                    eta_sec = remaining / max(rate, 1e-6)
                    eta_min = eta_sec / 60.0

                    print(
                        f"[heartbeat] split={split} "
                        f"done={processed+skipped}/{estimated_total_attempts} "
                        f"ok={processed} aug={augmented_written} skip={skipped} "
                        f"rate={rate:.2f} vid/s "
                        f"eta={eta_min:.1f} min "
                        f"storage={storage_gb:.2f}/{MAX_STORAGE_GB:.1f} GB"
                    )
                    last_heartbeat = now

        if uncovered_classes:
            if split == 'train' and train_stopped_for_val_budget:
                print(
                    f"[info] Stopped train early to preserve validation budget. "
                    f"Unprocessed train classes this pass: {len(uncovered_classes)}"
                )
            elif used_bytes >= max_bytes:
                raise RuntimeError(
                    f"Storage cap reached before covering all classes in split '{split}'. "
                    f"Missing classes: {sorted(uncovered_classes)[:10]}"
                )
            else:
                raise RuntimeError(
                    f"Preprocessing failed to produce any tensor for {len(uncovered_classes)} classes "
                    f"in split '{split}'. Examples: {sorted(uncovered_classes)[:10]}"
                )

        split_elapsed = time.time() - split_start
        print(
            f"[split summary] {split}: "
            f"covered={len(processed_by_class)}/{len(classes)} "
            f"ok={processed_by_split[split]} skip={skipped_by_split[split]} "
            f"time={split_elapsed/60:.1f} min"
        )

total_elapsed = time.time() - overall_start
print(f"Preprocessing complete: processed={processed}, skipped={skipped}, elapsed={total_elapsed/60:.1f} min")
print(f"Manifest: {manifest_path}")
print(f"Storage used: {used_bytes/(1024**3):.2f} GB / {MAX_STORAGE_GB:.1f} GB")
print(f"Augmented tensors written: {augmented_written}")

remaining_aug = sum(v for v in augment_remaining.values() if v > 0)
if ENABLE_MINORITY_AUGMENT:
    print(f"Remaining planned augmentations not written (budget/caps limited): {remaining_aug}")

if decode_error_types:
    print('Top decode failure types:')
    for k, v in decode_error_types.most_common(10):
        print(f'  {k}: {v}')

In [ ]:
npy_files = glob.glob(os.path.join(OUT_ROOT, '**', '*.npy'), recursive=True)
print(f'Total .npy files: {len(npy_files)}')
if npy_files:
    s = np.load(npy_files[0])
    print('shape:', s.shape)
    print('dtype:', s.dtype)
    print('min/max:', float(s.min()), float(s.max()))

for split in ('train', 'val'):
    cnt = len([p for p in npy_files if f'/{split}/' in p or f'\\{split}\\' in p])
    print(f'{split}: {cnt}')

print('Done. Save /kaggle/working as a Kaggle Dataset.')